In [6]:
from dotenv import load_dotenv
load_dotenv()
import logfire
import os
import json

logfire.configure(service_name="react_agent")

Logfire project URL: https://logfire-us.pydantic.dev/syednoormujassum/first-logfire

In [2]:
logfire.instrument_openai()

In [4]:
os.environ["Google_API"]=os.getenv("Gemini_API")

In [51]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

gemini_llm=ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API"),
    temperature=0.5,
    model="gemini-3.5-flash-lite"

)

openai_llm=ChatOpenAI(
    api_key=os.getenv("Open_AI_API"),
    model="gpt-6-luna"

)

In [52]:
from langchain_community. vectorstores import FAISS
from langchain_core.documents import Document
from langchain_community.document_loaders import json_loader

knowledge_base=os.getcwd()+'/document.json'


with open(knowledge_base) as f:
    data=json.load(f)


DOCS=[Document(metadata={"topic":raw["topic"], "source":raw["source"]},page_content=raw["content"])
for raw in data ]

print(f" Loaded {len(DOCS)} documents:{[d.metadata["topic"] for d in DOCS]}")

 Loaded 6 documents:['RAG', 'Finetuning', 'Evals', 'Observability', 'Prompting', 'LLM Gateways']


In [53]:

embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview",api_key=os.environ["Google_API"])

vector_store=FAISS.from_documents(DOCS,embeddings)
retriever=vector_store.as_retriever(search_kwargs={"k":2})

print("FAISS index ready ✅")

FAISS index ready ✅


In [54]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, HumanMessage,ToolMessage
from langchain.tools import tool

# Retreiver as a plain python tool

@tool
def search_knowledge_base(query:str)->str:
    """Search the knowledge base for LLM production topic: RAG, guardrails, gateways,
    observability, evaluation and fine-tuning"""

    docs=vector_store.similarity_search(query=query,k=3)
    return "\n\n".join(
            f"[{doc.metadata['topic']}] {doc.page_content}"
            for doc in docs
        )

In [55]:
agent = create_agent(
    model=openai_llm,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a helpful assistant. Use search_knowledge_base only when the question "
        "is about a topic covered by the local knowledge base: RAG, guardrails, LLM "
        "gateways, observability, evaluations, or fine-tuning. For unrelated general "
        "knowledge questions (for example, geography or capitals), answer directly "
        "without calling the tool. If you do use the tool, base the answer on its results."
    ),
)

In [56]:
# Trace agent runner

def run_agent(question:str, user_id:str ="anonymous"):
    with logfire.span("Agent Run", question=question, user_id=user_id):
        result=agent.invoke({"messages":[HumanMessage(content=question)]})

        ## Last messgae may be an AI message  or tool message with empty content
        # - iterate backwards to find last ai message  with actual text

        last_AI_msg=next(
            (m for m in reversed(result["messages"]) if isinstance (m, AIMessage)),None
        )

        answer= last_AI_msg.text if last_AI_msg else " "
        used_tool= any(isinstance(m, ToolMessage) for m in result["messages"])

        logfire.info("Agent_Done", used_tool=used_tool, answer_length=len(answer))
        return  answer, used_tool

In [57]:
queries=[
    ("What is LLM Gaurdrails","sid"),
    ("What are LLM gateways, how do they work","priya"),
    ("What are capital of Toronto","rajesh")
]

In [58]:
for q , uid in queries:
    print(f"{'='*55}")
    answer, used_tool = run_agent(question=q,user_id=uid)
    print(f"Question: {q}")
    print(f"Tools Used: {used_tool} {' <- retireved from Knowledge Base' if used_tool else '<- Retrieved directly'}")
    print(f"Answer: {answer[:300]}")

23:22:50.357 Agent Run
23:22:50.404   Responses API with 'gpt-6-luna' [LLM]
23:22:54.273   Responses API with 'gpt-6-luna' [LLM]
23:22:57.101   Responses API with 'gpt-6-luna' [LLM]
23:23:00.772   Agent_Done
Question: What is LLM Gaurdrails
Tools Used: True  <- retireved from Knowledge Base
Answer: **LLM guardrails** are safeguards around an AI application that help keep its inputs and outputs within defined rules.

For example, they can:
- Block harmful or disallowed requests
- Detect or redact sensitive information
- Check that responses follow a required format or policy
- Limit usage or ro
23:23:00.773 Agent Run
23:23:00.777   Responses API with 'gpt-6-luna' [LLM]
23:23:03.123   Responses API with 'gpt-6-luna' [LLM]
23:23:06.658   Agent_Done
Question: What are LLM gateways, how do they work
Tools Used: True  <- retireved from Knowledge Base
Answer: An **LLM gateway** is a service between your application and one or more LLM providers. Your app sends requests to the gateway rather t